# Clustering: KMeans, hierarchical, DBSCAN — Code-Along Template

**Type into this notebook during class.** The explanation is here; the code is not.

| | |
|---|---|
| **Follow along** | Write the code in each `# TODO` cell. The hint names the functions and arguments. |
| **If you fall behind** | Do not copy frantically. Skip to the next **CHECKPOINT**, run it, rejoin. |
| **If a cell errors** | Later cells depend on it and will fail too. Run the next CHECKPOINT rather than debugging under time pressure. |
| **After class** | The worked version is `D013_03_DBSCAN_Clustering.ipynb`. Compare, do not copy. |

Cells that are already filled in are deliberate — imports and quick glances at the
data cost class time and teach nothing.

# DBSCAN Clustering

## Learning Objectives
By the end of this notebook, you will:
- Understand density-based clustering concepts
- Know when DBSCAN is better than K-Means
- Master the eps and min_samples parameters
- Use the k-distance graph to find optimal eps
- Handle outliers/noise points with DBSCAN

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_blobs
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
# ═══ CHECKPOINT — "Why DBSCAN?" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_blobs
from sklearn.neighbors import NearestNeighbors
import warnings

print("Checkpoint restored. Ready for: Why DBSCAN?")

## 1. Why DBSCAN?

### K-Means Limitations
K-Means works well for **spherical, evenly-sized clusters**. But real data often has:
- Arbitrary-shaped clusters
- Clusters of different sizes/densities
- Outliers/noise

Let's see where K-Means fails:

In [ ]:
# TODO: compute `kmeans`, `kmeans_labels`, `dbscan`
# Name your result: kmeans, kmeans_labels, dbscan, dbscan_labels

## 2. How DBSCAN Works

### Core Concepts

DBSCAN groups points that are **densely packed together**, marking points in low-density regions as outliers.

### Key Parameters

| Parameter | Description |
|-----------|-------------|
| **eps** (ε) | Maximum distance between two points to be considered neighbors |
| **min_samples** | Minimum points required to form a dense region (core point) |

### Point Types

1. **Core Point**: Has at least `min_samples` neighbors within `eps` distance
2. **Border Point**: Within `eps` of a core point but has fewer than `min_samples` neighbors
3. **Noise Point**: Neither core nor border (labeled as -1)

In [ ]:
# TODO: compute `X_demo`, `dbscan_demo`, `labels_demo`
# Name your result: X_demo, dbscan_demo, labels_demo, core_samples

## 3. Finding Optimal eps: The K-Distance Graph

The most challenging part of DBSCAN is choosing the right `eps`. The **k-distance graph** helps:

1. For each point, calculate distance to its k-th nearest neighbor (k = min_samples)
2. Sort these distances
3. Plot them
4. Look for the "elbow" - where the curve bends sharply

In [ ]:
# TODO: compute `neighbors`, `k_distances`, `k_distances_sorted`
# Name your result: neighbors, k_distances, k_distances_sorted

## 4. Parameter Sensitivity

Let's see how different eps values affect clustering:

In [ ]:
# TODO: draw a `plt.subplots`
# Name your result: eps_values, dbscan, labels, n_clusters

In [ ]:
# ═══ CHECKPOINT — "DBSCAN vs K-Means: When to Use Each" ═══════════════════════════════════════════════
#
# Run this ONLY if you are behind, or a cell above threw an error and the rest
# stopped working. It rebuilds everything this section needs, so you can rejoin
# here instead of being stuck for the rest of the class.
#
# Running it when you are NOT stuck is harmless. If it leaves you stuck anyway,
# that is a bug in the template, not in your code — say so.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_moons, make_blobs
from sklearn.neighbors import NearestNeighbors
import warnings
X_moons, y_moons = make_moons(n_samples=300, noise=0.05, random_state=42)
kmeans = KMeans(n_clusters=2, random_state=42)
kmeans_labels = kmeans.fit_predict(X_moons)
dbscan = DBSCAN(eps=0.2, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_moons)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
X_demo = np.vstack([np.random.randn(50, 2) * 0.5 + [0, 0], np.random.randn(50, 2) * 0.5 + [3, 3], np.random.randn(5, 2) * 0.3 + [1.5, 1.5]])
dbscan_demo = DBSCAN(eps=0.5, min_samples=5)
labels_demo = dbscan_demo.fit_predict(X_demo)
core_samples = dbscan_demo.core_sample_indices_
colors = ['red' if l == -1 else 'blue' if l == 0 else 'green' for l in labels_demo]
noise_mask = labels_demo == -1
def find_optimal_eps(X, min_samples=5):
    """
    Use k-distance graph to find optimal eps.
    
    Parameters:
    -----------
    X : array-like
        Feature matrix
    min_samples : int
        min_samples parameter for DBSCAN
    
    Returns:
    --------
    distances : array
        Sorted k-distances
    """
    neighbors = NearestNeighbors(n_neighbors=min_samples)
    neighbors.fit(X)
    distances, indices = neighbors.kneighbors(X)
    k_distances = distances[:, -1]
    k_distances_sorted = np.sort(k_distances)
    return k_distances_sorted
k_distances = find_optimal_eps(X_demo, min_samples=5)
eps_values = [0.1, 0.3, 0.5, 0.8, 1.0]
fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, eps in zip(axes, eps_values):
    dbscan = DBSCAN(eps=eps, min_samples=5)
    labels = dbscan.fit_predict(X_demo)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = sum(labels == -1)
    colors = ['gray' if l == -1 else plt.cm.viridis(l / max(1, max(labels))) for l in labels]
    ax.scatter(X_demo[:, 0], X_demo[:, 1], c=colors)
    ax.set_title(f'eps={eps}\nClusters: {n_clusters}, Noise: {n_noise}')

print("Checkpoint restored. Ready for: DBSCAN vs K-Means: When to Use Each")

## 5. DBSCAN vs K-Means: When to Use Each

| Aspect | K-Means | DBSCAN |
|--------|---------|--------|
| **Cluster Shape** | Spherical only | Any shape |
| **Number of Clusters** | Must specify K | Auto-detected |
| **Outlier Handling** | Assigns to nearest cluster | Labels as noise (-1) |
| **Cluster Size** | Similar sizes work best | Handles varying sizes |
| **Speed** | Fast (O(n)) | Slower (O(n²) or O(n log n)) |
| **Parameters** | K | eps, min_samples |

In [ ]:
# TODO: compute `X_varied`, `scaler`, `X_varied_scaled`
# Name your result: X_varied, scaler, X_varied_scaled, kmeans

## 6. Practical Example: Customer Data with Outliers

In [ ]:
# TODO: compute `normal_customers`, `outliers`, `X_customers`
# Name your result: normal_customers, outliers, X_customers, scaler

## 7. Exercise: Apply DBSCAN to the Seeds Dataset

1. Load the seeds dataset
2. Scale the features
3. Use k-distance graph to find optimal eps
4. Apply DBSCAN
5. Compare results with K-Means

In [ ]:
# Load data
seeds = pd.read_csv('../../00-data/seeds.csv')
X_seeds = seeds.values

# Scale
scaler = StandardScaler()
X_seeds_scaled = scaler.fit_transform(X_seeds)

# YOUR CODE HERE
# 1. Plot k-distance graph to find optimal eps
# 2. Apply DBSCAN with your chosen eps and min_samples=5
# 3. Compare with K-Means (K=3)
# 4. Visualize using two features (e.g., area vs perimeter)



## Summary

### Key Takeaways:

1. **DBSCAN** finds clusters based on density, not distance to centroids
2. **eps** defines the neighborhood radius
3. **min_samples** defines the minimum density for a core point
4. Use **k-distance graph** to find optimal eps
5. DBSCAN naturally identifies **outliers** (labeled as -1)
6. DBSCAN doesn't require specifying the number of clusters

### When to Use DBSCAN:
- Arbitrary-shaped clusters
- Unknown number of clusters
- Data contains outliers
- Clusters have varying densities

---

**Next:** [04_Cluster_Evaluation_Metrics.ipynb](04_Cluster_Evaluation_Metrics.ipynb)